In [4]:
from numeric_qwen2_5_vl import NumericQwen2_5_VLForConditionalGeneration, NumericQwen2_5_VLProcessor
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [5]:
from numeric_qwen2_5_vl import NumericQwen2_5_VLForConditionalGeneration, NumericQwen2_5_VLProcessor
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
MODEL_PATH = "/data1/wangzhiye/qwen253B"
model = NumericQwen2_5_VLForConditionalGeneration.from_pretrained(
    # "Qwen/Qwen2.5-VL-3B-Instruct",
    # "/data1/wangzhiye/qwen253B",
    MODEL_PATH,
    # "/data1/wangzhiye/1a1a11/custom_qwen_checkpoint_4250/output/checkpoint-4250",
    device_map="auto")
# processor = NumericQwen2_5_VLProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

You are using a model of type qwen2_5_vl to instantiate a model of type numeric_qwen2_5_vl. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.81s/it]
Some weights of NumericQwen2_5_VLForConditionalGeneration were not initialized from the model checkpoint at /data1/wangzhiye/qwen253B and are newly initialized: ['numeric_embedding.0.bias', 'numeric_embedding.0.weight', 'numeric_embedding.2.bias', 'numeric_embedding.2.weight', 'numeric_embedding.3.bias', 'numeric_embedding.3.weight', 'regression_head.bias', 'regression_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
processor = NumericQwen2_5_VLProcessor.from_pretrained(MODEL_PATH,local_files_only=True)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [20]:
# from numeric_qwen2_5_vl import NumericQwen2_5_VLProcessor
# processor = NumericQwen2_5_VLProcessor.from_pretrained("/data1/wangzhiye/1a1a11/custom_qwen_checkpoint_4250/output/checkpoint-4250")
from transformers import AutoProcessor
raw_processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct",local_files_only=True)
from transformers import Qwen2_5_VLProcessor
raw_processor_2 = Qwen2_5_VLProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct",local_files_only=True)

In [21]:
print(hasattr(processor, "chat_template"))
print(processor.chat_template)
print(raw_processor.chat_template)
print(raw_processor_2.chat_template)

True
None
{% set image_count = namespace(value=0) %}{% set video_count = namespace(value=0) %}{% for message in messages %}{% if loop.first and message['role'] != 'system' %}<|im_start|>system
You are a helpful assistant.<|im_end|>
{% endif %}<|im_start|>{{ message['role'] }}
{% if message['content'] is string %}{{ message['content'] }}<|im_end|>
{% else %}{% for content in message['content'] %}{% if content['type'] == 'image' or 'image' in content or 'image_url' in content %}{% set image_count.value = image_count.value + 1 %}{% if add_vision_id %}Picture {{ image_count.value }}: {% endif %}<|vision_start|><|image_pad|><|vision_end|>{% elif content['type'] == 'video' or 'video' in content %}{% set video_count.value = video_count.value + 1 %}{% if add_vision_id %}Video {{ video_count.value }}: {% endif %}<|vision_start|><|video_pad|><|vision_end|>{% elif 'text' in content %}{{ content['text'] }}{% endif %}{% endfor %}<|im_end|>
{% endif %}{% endfor %}{% if add_generation_prompt %}<|im_s

In [9]:
def test_numeric_processor(numeric_processor):
 
    # 2. 准备含有数值标记的文本
    text_with_numbers = "这个产品评分为<num><8.5>分，价格是<num><299.99>元。"
    text_without_numbers = "这个产品评分为8.5分，价格是299.99元。"
    batch_texts = [
        "产品A评分为<num><9.2>分，他的价格为<num><599.99>元。",
        "产品B评分为<num><6.7>分，价格<num><199.5>元。",
        "产品C评分为<num><4.3>分。",
        "产品D评分为<4.3>分。"
    ]
    
    # 3. 处理单个文本
    print("===== 测试单个文本 =====")
    result = numeric_processor(text=text_with_numbers, return_tensors="pt")
    result_without_numbers = numeric_processor(text=text_without_numbers, return_tensors="pt")
    print(f"处理后的文本: {numeric_processor.tokenizer.decode(result['input_ids'][0])}")
    print(f"处理后的文本逐token输出:")
    for token_id in result['input_ids'][0]:
        print(numeric_processor.tokenizer.decode([token_id]), end='|')
    print()
    print(f"提取的数值: {result['numeric_values']}")
    print(f"提取的数值位置: {result['numeric_positions']}")
    print(f"处理后的文本（无数值标记）: {numeric_processor.tokenizer.decode(result_without_numbers['input_ids'][0])}")
    print(f"提取的数值（无数值标记）: {result_without_numbers['numeric_values']}")
    
    # 4. 处理批量文本
    print("\n===== 测试批量文本 =====")
    batch_result = numeric_processor(text=batch_texts, return_tensors="pt", padding=True)
    
    for i, text in enumerate(batch_texts):
        print(f"\n样本 {i+1}:")
        print(f"原始文本: {text}")
        print(f"处理后的文本: {numeric_processor.tokenizer.decode(batch_result['input_ids'][i])}")
    
    print(f"\n批量提取的数值: {batch_result['numeric_values']}")
    print(f"\n批量提取的数值位置: {batch_result['numeric_positions']}")
    
    return numeric_processor

# 执行测试
num_processor = test_numeric_processor(processor)

===== 测试单个文本 =====
处理后的文本: 这个产品评分为<num><num_pad>分，价格是<num><num_pad>元。
处理后的文本逐token输出:
这个|产品|评|分为|<num>|<num_pad>|分|，|价格|是|<num>|<num_pad>|元|。|
提取的数值: [[8.5, 299.99]]
提取的数值位置: [[5, 11]]
处理后的文本（无数值标记）: 这个产品评分为8.5分，价格是299.99元。
提取的数值（无数值标记）: [[]]

===== 测试批量文本 =====

样本 1:
原始文本: 产品A评分为<num><9.2>分，他的价格为<num><599.99>元。
处理后的文本: 产品A评分为<num><num_pad>分，他的价格为<num><num_pad>元。

样本 2:
原始文本: 产品B评分为<num><6.7>分，价格<num><199.5>元。
处理后的文本: 产品B评分为<num><num_pad>分，价格<num><num_pad>元。<|endoftext|><|endoftext|>

样本 3:
原始文本: 产品C评分为<num><4.3>分。
处理后的文本: 产品C评分为<num><num_pad>分。<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>

样本 4:
原始文本: 产品D评分为<4.3>分。
处理后的文本: 产品D评分为<4.3>分。<|endoftext|><|endoftext|><|endoftext|><|endoftext|>

批量提取的数值: [[9.2, 599.99], [6.7, 199.5], [4.3], []]

批量提取的数值位置: [[5, 12], [5, 10], [5], []]


/data1/wangzhiye/miniconda3/envs/llava/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:2696: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [18]:
conversation = processor.apply_chat_template(
    [
        {"role": "user", "content": "这个产品评分为<num><8.5>分，价格是<num><299.99>元。"},
        {"role": "assistant", "content": "好的，我已经记录下来了。"},
        {"role": "user", "content": "请问这个产品的评分是多少？"},
    ],
    add_generation_prompt=True,
    return_tensors="pt"
)

ValueError: Cannot use apply_chat_template because this processor does not have a chat template.